In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev
from matplotlib.animation import FuncAnimation
from matplotlib.widgets import Slider, Button

In [ ]:
%matplotlib widget

data = np.genfromtxt("20260710_141408.csv", dtype=float, delimiter=",", skip_header=1, missing_values="", filling_values=np.nan)

t = data[:, 1]

x_red = data[:, 2]
y_red = data[:, 3]

x_green = data[:, 4]
y_green = data[:, 5]

def fill_nan(t, x, y, points=5):
    x = x.copy()
    y = y.copy()

    for i in np.where(np.isnan(x) | np.isnan(y))[0]:

        valid = ~np.isnan(x) & ~np.isnan(y)

        before = np.where(valid[:i])[0][-points:]
        after = np.where(valid[i + 1:])[0][:points] + i + 1

        neighbours = np.concatenate((before, after))

        if len(neighbours) >= 2:
            x[i] = np.interp(t[i], t[neighbours], x[neighbours])
            y[i] = np.interp(t[i], t[neighbours], y[neighbours])

    return x, y


x_red, y_red = fill_nan(t, x_red, y_red)
x_green, y_green = fill_nan(t, x_green, y_green)


def smooth_trajectory(t, x, y, smooth=2):
    valid = ~np.isnan(t) & ~np.isnan(x) & ~np.isnan(y)

    t = t[valid]
    x = x[valid]
    y = y[valid]

    _, idx = np.unique(t, return_index=True)

    t = t[idx]
    x = x[idx]
    y = y[idx]

    u = (t - t[0]) / (t[-1] - t[0])

    tck, _ = splprep([x, y], u=u, s=smooth, k=3)

    u_smooth = np.linspace(0, 1, len(t) * 5)
    x_smooth, y_smooth = splev(u_smooth, tck)

    return np.array(x_smooth), np.array(y_smooth)


red_x, red_y = smooth_trajectory(t, x_red, y_red)
green_x, green_y = smooth_trajectory(t, x_green, y_green)

n = len(red_x)

fig, ax = plt.subplots()
plt.subplots_adjust(bottom=0.25)

ax.set_xlim(min(red_x.min(), green_x.min()), max(red_x.max(), green_x.max()))

ax.set_ylim(min(red_y.min(), green_y.min()), max(red_y.max(), green_y.max()))

ax.set_aspect("equal")

red_line, = ax.plot([], [], "r-", lw=2)
green_line, = ax.plot([], [], "g-", lw=2)
blue_line, = ax.plot([], [], "b-", lw=2)

red_point, = ax.plot([], [], "ro")
green_point, = ax.plot([], [], "go")

red_trails = []
green_trails = []

trail_length = 360

for i in range(trail_length):
    red_trails.append(ax.plot([], [], "r-", lw=2, alpha=(i + 1) / trail_length)[0])
    green_trails.append(ax.plot([], [], "g-", lw=2, alpha=(i + 1) / trail_length)[0])

# Slider
slider_ax = plt.axes([0.15, 0.10, 0.70, 0.03])

frame_slider = Slider(slider_ax, "Frame", 0, len(t) - 1, valinit=0, valstep=1)


# Play / Pause button
button_ax = plt.axes([0.43, 0.04, 0.14, 0.04])
play_button = Button(button_ax, "Play")

playing = False
frame = 0


def draw_frame(frame_number):
    global frame

    frame = int(frame_number)

    end = int(frame / (len(t) - 1) * (n - 1))

    for i in range(trail_length):
        start = max(0, end - trail_length + i)
        stop = max(0, end - trail_length + i + 2)

        red_trails[i].set_data(red_x[start:stop], red_y[start:stop])
        green_trails[i].set_data(green_x[start:stop], green_y[start:stop])

    blue_line.set_data([red_x[end], green_x[end]], [red_y[end], green_y[end]])

    red_point.set_data([red_x[end]], [red_y[end]])
    green_point.set_data([green_x[end]], [green_y[end]])

    fig.canvas.draw_idle()


def slider_update(value):
    draw_frame(value)


frame_slider.on_changed(slider_update)


def play_pause(event):
    global playing

    playing = not playing

    if playing:
        play_button.label.set_text("Pause")
    else:
        play_button.label.set_text("Play")


play_button.on_clicked(play_pause)


def animate(_):
    global frame

    if not playing:
        return

    frame += 1

    if frame >= len(t):
        frame = 0

    # Update graph directly
    draw_frame(frame)

    # Update slider without triggering its callback
    frame_slider.eventson = False
    frame_slider.set_val(frame)
    frame_slider.eventson = True


ani = FuncAnimation(fig, animate, interval=720, cache_frame_data=False)

draw_frame(0)

plt.show()